# MobileNetV3-Large performance evaluation

This notebook reproduces the frozen `transfer-mobile-v1` evaluation on the 592-image `parcel_binary_v2` test split. It does not train the model or change its threshold.

### What those words mean

- **MobileNetV3-Large** is the image-classification model. It looks at the whole image and predicts either `damaged` or `intact`.
- **Frozen model** means the saved weights and decision threshold are loaded exactly as they are. Nothing is learned or updated here.
- **Test split** is the held-out set used to measure how well the finished model generalizes.
- **Evaluation** means making predictions and comparing them with the known labels.

### What you will get

The notebook prints the main performance numbers and creates a confusion matrix, ROC curve, precision-recall curve, prediction CSV, and contact sheets containing mistakes. At the end, it downloads all generated files as one ZIP archive.

Before starting:

1. In Colab, choose **Runtime → Change runtime type → T4 GPU**.
2. Put the complete project folder in Google Drive. The folder must include `datasets/processed/parcel_binary_v2`, the frozen checkpoint, the freeze manifest, and `scripts/evaluate_transfer_model_final.py`.
3. Update `PROJECT_ROOT` in the setup cell if your Drive folder has a different name.

> This rerun is for reproducibility. The original certified result remains the permanent result reported in `reports/PHASE-11_transfer_model_final_test.md`.

In [ ]:
# Google Drive is used because the complete image dataset is too large for
# the small portable demo bundle. Colab will ask you to authorize access.
from google.colab import drive

drive.mount('/content/drive')

## 1. Locate the complete project

In [ ]:
# Path makes file and folder paths easier to construct safely.
from pathlib import Path
import os

# Change this path if your project has another folder name in Google Drive.
PROJECT_ROOT = Path('/content/drive/MyDrive/Project_Card_board')

# Stop immediately with a useful message if the folder name is wrong.
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(
        f'Project folder not found: {PROJECT_ROOT}\n'
        'Upload the complete project to Drive or update PROJECT_ROOT.'
    )

# Make the project folder the current working directory. Relative paths such
# as 'scripts/...' and 'models/...' will now point to the correct files.
os.chdir(PROJECT_ROOT)
print('Project root:', Path.cwd())

## 2. Check the runtime and required files

In [ ]:
# Import the libraries used by the evaluation script. These are normally
# already available in a standard Google Colab runtime.
import sys
import torch
import torchvision
import sklearn
import matplotlib

print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('torchvision:', torchvision.__version__)
print('scikit-learn:', sklearn.__version__)
print('matplotlib:', matplotlib.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

# Verify every important input before spending time running inference.
# The manifest contains labels and integrity hashes; the checkpoint contains
# the learned model weights; the JSON file contains the frozen threshold.
required_paths = [
    Path('scripts/evaluate_transfer_model_final.py'),
    Path('datasets/processed/parcel_binary_v2/manifests/dataset_manifest.csv'),
    Path('datasets/processed/parcel_binary_v2/manifests/audit_summary.json'),
    Path('models/selected_transfer_model/selected_transfer_model.json'),
    Path('models/transfer_learning/mobilenet_v3_large_frozen/best_model.pt'),
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required project files:\n- ' + '\n- '.join(missing))
print('Preflight check passed.')

## 3. Run the frozen final evaluation

A timestamped output directory is used so rerunning this notebook never overwrites an earlier result. The script verifies the checkpoint, dataset manifest, file hashes, split isolation, frozen threshold, and expected test size before inference.

In [ ]:
from datetime import datetime, timezone

# Add the current UTC date and time to the folder name. This prevents a new
# run from overwriting results produced by an earlier run.
run_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
MOBILENET_OUTPUT = Path('/content/results') / f'mobilenet_final_{run_stamp}'
MOBILENET_OUTPUT.mkdir(parents=True, exist_ok=False)
print('Results will be saved to:', MOBILENET_OUTPUT)

In [ ]:
# subprocess runs the existing project evaluation script exactly as if you
# typed the command in a terminal. check=True makes Colab stop on an error.
import subprocess

command = [
    sys.executable,
    'scripts/evaluate_transfer_model_final.py',
    '--dataset', 'datasets/processed/parcel_binary_v2',
    '--freeze-manifest', 'models/selected_transfer_model/selected_transfer_model.json',
    '--output', str(MOBILENET_OUTPUT),
    '--batch-size', '32',
]
print('Running:', ' '.join(command))
subprocess.run(command, check=True)
print('Evaluation finished successfully.')

## 4. Display the metrics and plots

Here is a beginner-friendly guide to the metrics:

- **Accuracy:** percentage of all images classified correctly.
- **Damaged recall:** percentage of truly damaged parcels detected. High recall means fewer damaged parcels are missed.
- **Damaged precision:** percentage of `damaged` predictions that are actually damaged.
- **F1:** a balance between precision and recall.
- **Specificity:** percentage of intact parcels correctly recognized as intact.
- **ROC-AUC / PR-AUC:** overall ranking quality across many possible thresholds; values closer to 1 are better.
- **Threshold:** the fixed damaged-probability cutoff. It is loaded, not selected by this notebook.
- **Latency:** average model-forward time for one image in this run. Colab hardware and warm-up can change it.

In [ ]:
import json
import pandas as pd
from IPython.display import Image as DisplayImage, display

# Read the machine-readable JSON generated by the evaluation script.
metrics = json.loads((MOBILENET_OUTPUT / 'final_test_metrics.json').read_text())
summary = {
    'Test images': metrics['test_images'],
    'Accuracy': metrics['accuracy'],
    'Damaged recall': metrics['damaged_recall'],
    'Damaged precision': metrics['damaged_precision'],
    'Damaged F1': metrics['damaged_f1'],
    'Specificity': metrics['specificity'],
    'ROC-AUC': metrics['roc_auc'],
    'PR-AUC': metrics['pr_auc'],
    'Threshold': metrics['frozen_threshold'],
    'Latency (ms/image)': metrics['average_forward_latency_ms_per_image'],
}
# A DataFrame displays the selected values as a clean two-column table.
display(pd.DataFrame(summary.items(), columns=['Metric', 'Value']))

# Display each plot directly below the metric table.
for filename in ['final_confusion_matrix.png', 'final_roc_curve.png', 'final_pr_curve.png']:
    path = MOBILENET_OUTPUT / filename
    if path.exists():
        display(DisplayImage(filename=str(path)))

## 5. Inspect errors and download all results

A **false negative** is a damaged parcel predicted as intact. A **false positive** is an intact parcel predicted as damaged. The contact sheets help you understand the visual cases the model finds difficult.

In [ ]:
# Show up to 50 false negatives and 50 false positives. These images are for
# error analysis only; they must not be used to tune this frozen test result.
for filename in ['final_false_negative_contact_sheet.jpg', 'final_false_positive_contact_sheet.jpg']:
    path = MOBILENET_OUTPUT / filename
    if path.exists():
        display(DisplayImage(filename=str(path)))

In [ ]:
# Compress every JSON, CSV, plot, and contact sheet into one ZIP file.
import shutil
from google.colab import files

archive = shutil.make_archive(str(MOBILENET_OUTPUT), 'zip', root_dir=MOBILENET_OUTPUT)
print('Created:', archive)
files.download(archive)